# Etapa 13 — Teste de sanidade do Grad-CAM
Decisões fixadas ANTES de rodar: amostra de 50 imagens do teste (seed 42); pesos de todos os módulos da `layer4` realeatorizados (seed 42); classe-alvo = classe prevista pelo modelo ORIGINAL, nos dois mapas; métrica = correlação de Spearman por imagem (mapa original x mapa com pesos aleatórios).

In [ ]:
!pip install -q grad-cam==1.5.7
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, random, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torchvision
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from torchvision.models import resnet18

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

BASE = '/content/drive/MyDrive/xai-brain-tumor'
paths = dict(
    ckpt=f'{BASE}/models/resnet18_B_best.pt',
    cache=f'{BASE}/data/cache_224.npz',
    test=f'{BASE}/data/splits/test.csv',
    preds=f'{BASE}/results/metrics/predicoes_teste.csv',
)
for k, p in paths.items():
    print('OK   ' if os.path.exists(p) else 'FALTA', p)
assert all(os.path.exists(p) for p in paths.values()), 'Algum arquivo nao foi encontrado (veja FALTA acima)'

In [ ]:
nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}

z = np.load(paths['cache'])
X, M, Y, IDS = z['X'], z['M'], z['Y'], z['IDS']
row_of = {int(i): k for k, i in enumerate(IDS)}
test_ids = pd.read_csv(paths['test'])['id'].astype(int).tolist()
print('imagens no teste:', len(test_ids))

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def prep(ids):
    # mesmo pre-processamento do teste: 1 canal -> 3 canais, normalizacao ImageNet, sem augmentation
    x = torch.from_numpy(np.stack([X[row_of[i]] for i in ids]).astype(np.float32))
    x = x.unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD

def build_model():
    m = resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 3)
    return m.to(device)

try:
    ckpt = torch.load(paths['ckpt'], map_location=device)
except Exception as e:
    print('torch.load padrao falhou, tentando weights_only=False:', repr(e)[:200])
    ckpt = torch.load(paths['ckpt'], map_location=device, weights_only=False)
print('chaves do checkpoint:', list(ckpt.keys()))
print('estrategia:', ckpt.get('estrategia'), '| epoca:', ckpt.get('epoca'), '| val macro-F1:', ckpt.get('val_macro_f1'))

model = build_model()
model.load_state_dict(ckpt['model'])
model.eval()

# Verificacao: o modelo carregado reproduz as previsoes da Etapa 9?
pred_orig = {}
with torch.no_grad():
    for s in range(0, len(test_ids), 64):
        b = test_ids[s:s + 64]
        out = model(prep(b).to(device)).argmax(1).cpu().numpy()
        pred_orig.update({i: int(p) for i, p in zip(b, out)})
ref = pd.read_csv(paths['preds'])
n_dif = sum(nomes[pred_orig[int(i)]] != p for i, p in zip(ref.id, ref.previsto))
print('previsoes diferentes da Etapa 9:', n_dif, 'de', len(ref))
assert n_dif == 0, 'O modelo carregado NAO reproduz a Etapa 9. Pare e avise.'

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

N_AMOSTRA = 50
rng = np.random.default_rng(SEED)
amostra = sorted(int(i) for i in rng.choice(sorted(test_ids), size=N_AMOSTRA, replace=False))
alvos = {i: pred_orig[i] for i in amostra}   # classe-alvo = prevista pelo modelo ORIGINAL
print('amostra (ids):', amostra)
print('classes reais na amostra:', pd.Series([nomes[int(Y[row_of[i]])] for i in amostra]).value_counts().to_dict())

def mapas(m, ids, alvos, bs=10):
    out = {}
    with GradCAM(model=m, target_layers=[m.layer4[-1]]) as cam:
        for s in range(0, len(ids), bs):
            b = ids[s:s + bs]
            r = cam(input_tensor=prep(b).to(device), targets=[ClassifierOutputTarget(alvos[i]) for i in b])
            for i, mp in zip(b, r):
                out[i] = mp.astype(np.float32)
    return out

orig = mapas(model, amostra, alvos)

# Conferencia informativa com os mapas salvos pelo colega (se existirem no Drive)
difs = []
for i in amostra:
    f = f'{BASE}/results/gradcam/{i}_pred.npy'
    if os.path.exists(f):
        difs.append(float(np.abs(np.load(f) - orig[i]).max()))
print('mapas salvos encontrados:', len(difs), 'de', N_AMOSTRA,
      '| maior diferenca absoluta vs recalculado:', max(difs) if difs else 'n/a')

In [ ]:
# Aleatorizar os pesos de TODOS os modulos da layer4 (conv e batchnorm); o resto da rede fica igual
rand_model = copy.deepcopy(model)
torch.manual_seed(SEED)
n_reset = 0
for m in rand_model.layer4.modules():
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters(); n_reset += 1
rand_model.eval()
print('modulos reiniciados na layer4:', n_reset)

w0, w1 = model.layer4[0].conv1.weight, rand_model.layer4[0].conv1.weight
print('layer4[0].conv1 mudou:', not torch.equal(w0, w1))
print('layer1[0].conv1 igual (nao deveria mudar):', torch.equal(model.layer1[0].conv1.weight, rand_model.layer1[0].conv1.weight))
print('fc igual (nao deveria mudar):', torch.equal(model.fc.weight, rand_model.fc.weight))

rand = mapas(rand_model, amostra, alvos)

with torch.no_grad():
    pred_rand = {}
    for s in range(0, len(amostra), 25):
        b = amostra[s:s + 25]
        out = rand_model(prep(b).to(device)).argmax(1).cpu().numpy()
        pred_rand.update({i: int(p) for i, p in zip(b, out)})

rows = []
for i in amostra:
    a, b = orig[i].ravel(), rand[i].ravel()
    rho = np.nan if (a.std() == 0 or b.std() == 0) else float(spearmanr(a, b)[0])
    rows.append(dict(id=i, real=nomes[int(Y[row_of[i]])], previsto_orig=nomes[pred_orig[i]],
                     previsto_rand=nomes[pred_rand[i]], spearman=rho))
res = pd.DataFrame(rows)

v = res.spearman.dropna()
resumo = dict(
    n_amostra=N_AMOSTRA, n_validos=int(len(v)), n_nan=int(res.spearman.isna().sum()),
    spearman_media=float(v.mean()), spearman_mediana=float(v.median()), spearman_dp=float(v.std()),
    spearman_min=float(v.min()), spearman_max=float(v.max()),
    previsao_rand_igual_orig=int((res.previsto_orig == res.previsto_rand).sum()),
)
print(res.round(3).to_string(index=False))
print()
print(json.dumps(resumo, indent=2, ensure_ascii=False))

In [ ]:
import importlib.metadata as md_
OUT = f'{BASE}/results/sanidade'
os.makedirs(OUT, exist_ok=True)

res.to_csv(f'{OUT}/sanidade_spearman.csv', index=False)
cfg = dict(
    seed=SEED, n_amostra=N_AMOSTRA, amostra_ids=amostra,
    amostragem='rng default_rng(42).choice sobre ids do teste ordenados, sem reposicao',
    camada_aleatorizada='todos os modulos com reset_parameters dentro de layer4',
    modulos_reiniciados=n_reset, camada_grad_cam='layer4[-1]',
    classe_alvo='classe prevista pelo modelo original, a mesma nos dois mapas',
    metrica='correlacao de Spearman por imagem entre mapas 224x224 achatados',
    checkpoint='resnet18_B_best.pt',
    versoes=dict(python=sys.version.split()[0], torch=torch.__version__, torchvision=torchvision.__version__,
                 numpy=np.__version__, scipy=__import__('scipy').__version__, grad_cam=md_.version('grad-cam')),
    resumo=resumo,
)
json.dump(cfg, open(f'{OUT}/sanidade_resumo.json', 'w'), indent=2, ensure_ascii=False)

ex = amostra[:6]
fig, axs = plt.subplots(2, 6, figsize=(18, 6.4))
for c, i in enumerate(ex):
    k = row_of[i]
    for r, (mp, tit) in enumerate([(orig[i], 'original'), (rand[i], 'layer4 aleatorizada')]):
        ax = axs[r, c]
        ax.imshow(X[k].astype(np.float32), cmap='gray')
        ax.imshow(mp, cmap='jet', alpha=0.4, vmin=0, vmax=1)
        ax.contour(M[k].astype(float), levels=[0.5], colors='white', linewidths=1)
        rho = res.loc[res.id == i, 'spearman'].iloc[0]
        ax.set_title(f'id {i} | {tit}' + (f'\nSpearman = {rho:.2f}' if r == 1 else ''), fontsize=9)
        ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/sanidade_exemplos.png', dpi=150)
plt.show()
print('Salvo em:', OUT, '->', sorted(os.listdir(OUT)))